    # -Common Table Expressions (CTEs)

Very important in interviews and real projects.

CTEs are basically:

named temporary result sets

Cleaner alternative to derived tables.

<span style="color:red">CTE — Q1

Create a CTE named:

high_salary

From Employees:

select employees with salary > 80000

Then display:

first_name
salary

from the CTE.</span>

In [1]:
WITH high_salary AS(
        SELECT
            first_name,
            salary
        FROM Employees
        WHERE salary > 80000
) SELECT * FROM high_salary

(3 rows affected)

first_name | salary  
-----------+---------
Amit       | 90000.00
Priya      | 85000.00
Vikram     | 95000.00
(3 rows)

Total execution time: 00:00:00.063

    -CTE with Aggregation

This is where CTEs become genuinely useful.

<span style="color: red">CTE — Q2

Create a CTE named:

dept_avg

From Employees:

calculate average salary department-wise

Then display only departments where:

avg_salary > 70000

Show:

department_id
avg_salary </span>

In [ ]:
WITH dept_avg AS(
    SELECT
        department_id,
        AVG(salary) AS avg_sal
    FROM Employees
    GROUP BY department_id
) 
SELECT * 
FROM dept_avg
WHERE avg_sal > 70000

(4 rows affected)

department_id | avg_sal     
--------------+-------------
1             | 82500.000000
2             | 80000.000000
3             | 80000.000000
5             | 71500.000000
(4 rows)

Total execution time: 00:00:01.016

    -Recursive CTE

Advanced topic.
Important for hierarchical data.

Used for:

employee-manager hierarchy
organization trees
category trees
parent-child structures
Core Idea

Recursive CTE references itself.

Structure has:

Anchor query
Recursive query

<span style="color:red">Recursive CTE — Q1

From Employees:

Write recursive CTE to display:

employee_id
first_name
manager_id

Start from employees whose:

manager_id IS NULL</span>

In [9]:
WITH cte_emp_details AS(
    -- Anchor Query
    SELECT 
        employee_id,
        first_name,
        manager_id
    FROM employees
)
select * 
from cte_emp_details
where manager_id is null

(3 rows affected)

employee_id | first_name | manager_id
------------+------------+-----------
101         | Amit       | NULL      
102         | Priya      | NULL      
107         | Vikram     | NULL      
(3 rows)

Total execution time: 00:00:01.004

<span style="color:red">Recursive CTE — Q1

Goal:
Show only top-level employees.

Meaning:

manager_id IS NULL

Create CTE:

emp_hierarchy

Display:

employee_id
first_name
manager_id

This is only anchor query practice.
No recursion yet.</span>

In [1]:
WITH emp_hierarchy AS(
    SELECT
        employee_id,
        first_name,
        manager_id
    FROM Employees
    WHERE manager_id is null  
)
SELECT * from emp_hierarchy


(3 rows affected)

employee_id | first_name | manager_id
------------+------------+-----------
101         | Amit       | NULL      
102         | Priya      | NULL      
107         | Vikram     | NULL      
(3 rows)

Total execution time: 00:00:00.074

<span style="color:red">Recursive CTE — Q2

Goal:

start from top-level employees
then find employees reporting to them

Create recursive CTE:

emp_hierarchy

Display:

employee_id
first_name
manager_id</span>

In [ ]:
WITH emp_hierarchy_recursive AS (
    -- Anchor Query
    SELECT
        employee_id,
        first_name,
        manager_id
    FROM Employees
    WHERE manager_id IS NULL

    UNION ALL

    -- Recursive Query
    SELECT
        e.employee_id,
        e.first_name,
        e.manager_id
    FROM Employees e
    JOIN emp_hierarchy_recursive eh 
    on e.manager_id = eh.employee_id  
)
SELECT * 
FROM emp_hierarchy_recursive

(10 rows affected)

employee_id | first_name | manager_id
------------+------------+-----------
101         | Amit       | NULL      
102         | Priya      | NULL      
107         | Vikram     | NULL      
104         | Sneha      | 102       
106         | Neha       | 102       
110         | Meera      | 102       
109         | Karan      | 106       
103         | Rahul      | 101       
105         | Arjun      | 101       
108         | Pooja      | 105       
(10 rows)

Total execution time: 00:00:00.030

<span style="color:red">Recursive CTE — Q3

Create recursive CTE:

emp_tree

Display:

employee_id
first_name
manager_id
hierarchy_level

Requirements:

top-level employees should have level = 1
increase level by 1 for each recursion level</span>

In [5]:
WITH emp_tree AS(
        -- Anchor Query 
        SELECT
            employee_id,
            first_name,
            manager_id,
            1 AS hierarchy_level
        FROM Employees
        WHERE manager_id IS NULL

        UNION ALL

        -- Recursive Query
        SELECT
            e.employee_id,
            e.first_name,
            e.manager_id,
            et.hierarchy_level + 1
        FROM Employees e 
        JOIN emp_tree AS et 
        on e.manager_id = et.employee_id

)
SELECT * 
FROM emp_tree


(10 rows affected)

employee_id | first_name | manager_id | hierarchy_level
------------+------------+------------+----------------
101         | Amit       | NULL       | 1              
102         | Priya      | NULL       | 1              
107         | Vikram     | NULL       | 1              
104         | Sneha      | 102        | 2              
106         | Neha       | 102        | 2              
110         | Meera      | 102        | 2              
109         | Karan      | 106        | 3              
103         | Rahul      | 101        | 2              
105         | Arjun      | 101        | 2              
108         | Pooja      | 105        | 3              
(10 rows)

Total execution time: 00:00:00.125

<span style="color:red">Recursive CTE — Q4

Create recursive CTE:

emp_path

Display:

employee_id
first_name
manager_id
hierarchy_path

Requirements:

top-level employee path should start with their own name
child employees should append their name to parent path

Example:

Amit -> Rahul -> Neha</span>

In [ ]:
WITH emp_path AS(
    -- Anchor Query 
    SELECT
        employee_id,
        first_name,
        manager_id,
        CAST(first_name AS varchar(MAX)) AS hierarchy_path
    FROM Employees
    where manager_id is null

    UNION ALL

    -- Recursive Query

    SELECT
        e.employee_id,
        e.first_name,
        e.manager_id,
        ep.hierarchy_path + '->' + e.first_name
    FROM Employees e
    JOIN emp_path ep
    ON e.manager_id = ep.employee_id  
)
SELECT * 
FROM emp_path



(10 rows affected)

employee_id | first_name | manager_id | hierarchy_path    
------------+------------+------------+-------------------
101         | Amit       | NULL       | Amit              
102         | Priya      | NULL       | Priya             
107         | Vikram     | NULL       | Vikram            
104         | Sneha      | 102        | Priya->Sneha      
106         | Neha       | 102        | Priya->Neha       
110         | Meera      | 102        | Priya->Meera      
109         | Karan      | 106        | Priya->Neha->Karan
103         | Rahul      | 101        | Amit->Rahul       
105         | Arjun      | 101        | Amit->Arjun       
108         | Pooja      | 105        | Amit->Arjun->Pooja
(10 rows)

Total execution time: 00:00:01.006

<span style="color:red">Recursive CTE — Q5

Create recursive CTE:

emp_levels

Display:

employee_id
first_name
manager_id
hierarchy_level

Requirements:

start from top-level employees
show only employees whose hierarchy level is:
<= 3</span>

In [15]:
WITH emp_levels AS(
    -- Anchor Query
    SELECT 
        employee_id,
        first_name,
        manager_id,
        1 AS hierarchy_level
    FROM Employees
    WHERE manager_id IS NULL


    UNION ALL 

    -- Recursive Query 

    SELECT
        e.employee_id,
        e.first_name,
        e.manager_id,
        el.hierarchy_level + 1
    FROM employees e 
    JOIN emp_levels el 
    ON e.manager_id = el.employee_id
    

)
SELECT * 
FROM emp_levels
WHERE hierarchy_level <= 3

(10 rows affected)

employee_id | first_name | manager_id | hierarchy_level
------------+------------+------------+----------------
101         | Amit       | NULL       | 1              
102         | Priya      | NULL       | 1              
107         | Vikram     | NULL       | 1              
104         | Sneha      | 102        | 2              
106         | Neha       | 102        | 2              
110         | Meera      | 102        | 2              
109         | Karan      | 106        | 3              
103         | Rahul      | 101        | 2              
105         | Arjun      | 101        | 2              
108         | Pooja      | 105        | 3              
(10 rows)

Total execution time: 00:00:00.570

<span style="color:red">Recursive CTE — Q6

Create recursive CTE:

emp_count

Display:

employee_id
first_name
manager_id
hierarchy_level

Requirements:

start from top-level employees
hierarchy level should increase recursively
show only employees whose:
hierarchy_level >= 2</span>

In [16]:
WITH emp_count AS(
    -- Anchor Query 
    SELECT
        employee_id,
        first_name,
        manager_id,
        1 AS hierarchy_level
    FROM employees
    WHERE manager_id is null


    UNION ALL

    -- Recursive Query 

    SELECT
        e.employee_id,
        e.first_name,
        e.manager_id,
        ec.hierarchy_level + 1
    FROM Employees e    
    JOIN emp_count ec 
    ON e.manager_id = ec.employee_id
)
select * 
from emp_count 
WHERE hierarchy_level >= 2

(7 rows affected)

employee_id | first_name | manager_id | hierarchy_level
------------+------------+------------+----------------
104         | Sneha      | 102        | 2              
106         | Neha       | 102        | 2              
110         | Meera      | 102        | 2              
109         | Karan      | 106        | 3              
103         | Rahul      | 101        | 2              
105         | Arjun      | 101        | 2              
108         | Pooja      | 105        | 3              
(7 rows)

Total execution time: 00:00:01.016

<span style="color:red">Recursive CTE — Q7

Create recursive CTE:

emp_chain

Display:

employee_id
first_name
manager_id
hierarchy_level
reporting_chain

Requirements:

top-level employee chain starts with their own name
recursively append child employee names
hierarchy level should increase recursively

Format example:

Amit -> Rahul -> Neha</span>

In [19]:
with emp_chain as (
    -- Anchor Query 
    select 
        employee_id,
        first_name,
        manager_id,
        1 as hierarchy_level,
        cast(first_name as varchar(max)) as reporting_chain
    from employees
    where manager_id is null


    union all 

    -- Recursive Query
    select 
        e.employee_id,
        e.first_name,
        e.manager_id,
        ech.hierarchy_level + 1,
        ech.reporting_chain + '->' + e.first_name
    from employees e    
    join emp_chain ech 
    on e.manager_id = ech.employee_id 
)
select *
from emp_chain

(10 rows affected)

employee_id | first_name | manager_id | hierarchy_level | reporting_chain   
------------+------------+------------+-----------------+-------------------
101         | Amit       | NULL       | 1               | Amit              
102         | Priya      | NULL       | 1               | Priya             
107         | Vikram     | NULL       | 1               | Vikram            
104         | Sneha      | 102        | 2               | Priya->Sneha      
106         | Neha       | 102        | 2               | Priya->Neha       
110         | Meera      | 102        | 2               | Priya->Meera      
109         | Karan      | 106        | 3               | Priya->Neha->Karan
103         | Rahul      | 101        | 2               | Amit->Rahul       
105         | Arjun      | 101        | 2               | Amit->Arjun       
108         | Pooja      | 105        | 3               | Amit->Arjun->Pooja
(10 rows)

Total execution time: 00:00:00.906

<span style="color: red">Recursive CTE — Q8

Create recursive CTE:

mgr_hierarchy

Display:

employee_id
first_name
manager_id
hierarchy_level

Requirements:

start from top-level employees
recursively traverse hierarchy
show only employees whose:
manager_id IS NOT NULL</span>

In [21]:
with mgr_hierarchy as(
    -- Anchor Query
    select 
        employee_id,
        first_name,
        manager_id,
        1 as hierarchy_level
    from employees
    where manager_id is null

    union all 

    -- Recursive Query

    select 
        e.employee_id,
        e.first_name,
        e.manager_id,
        mh.hierarchy_level + 1
    from employees e    
    join mgr_hierarchy mh 
    on e.manager_id = mh.employee_id
)
select * 
from mgr_hierarchy
where manager_id is not null

(7 rows affected)

employee_id | first_name | manager_id | hierarchy_level
------------+------------+------------+----------------
104         | Sneha      | 102        | 2              
106         | Neha       | 102        | 2              
110         | Meera      | 102        | 2              
109         | Karan      | 106        | 3              
103         | Rahul      | 101        | 2              
105         | Arjun      | 101        | 2              
108         | Pooja      | 105        | 3              
(7 rows)

Total execution time: 00:00:01.016

<span style="color:red">Recursive CTE — Q9

Create recursive CTE:

emp_depth

Display:

employee_id
first_name
manager_id
hierarchy_level

Requirements:

start from top-level employees
recursively build hierarchy
display only the deepest hierarchy level employees</span>

In [5]:
WITH emp_depth AS (
    -- Anchor Query 
    SELECT
        employee_id, 
        first_name,
        manager_id,
        1 as hierarchy_level
    FROM Employees
    where manager_id is null

    union all 

    -- Recursive Query 

    SELECT
        e.employee_id,
        e.first_name,
        e.manager_id,
        ed.hierarchy_level + 1
    FROM Employees e    
    join emp_depth ed  
    on e.manager_id = ed.employee_id
)
select * 
from emp_depth 
where hierarchy_level = (
    SELECT MAX(hierarchy_level)
    from emp_depth
)

(2 rows affected)

employee_id | first_name | manager_id | hierarchy_level
------------+------------+------------+----------------
109         | Karan      | 106        | 3              
108         | Pooja      | 105        | 3              
(2 rows)

Total execution time: 00:00:01.004

<span style="color:red">Recursive CTE — Q10

Create recursive CTE:

emp_manager_chain

Display:

employee_id
first_name
manager_id
hierarchy_level
manager_chain

Requirements:

start from top-level employees
hierarchy level should increase recursively
manager chain should contain only manager names</span>

In [7]:
with emp_manager_chain as(  
    -- Anchor Query 

    select 
        employee_id,
        first_name,
        manager_id,
        1 as hierarchy_level,
        cast(first_name as varchar(max)) as manager_chain
    from employees 
    where manager_id is null

    union all

    -- Recursive Query 

    select 
        e.employee_id,
        e.first_name,
        e.manager_id,
        emc.hierarchy_level + 1,
        emc.manager_chain + '->' + e.first_name
    from employees e
    join emp_manager_chain emc  
    on e.manager_id = emc.employee_id
)
select * 
from emp_manager_chain

(10 rows affected)

employee_id | first_name | manager_id | hierarchy_level | manager_chain     
------------+------------+------------+-----------------+-------------------
101         | Amit       | NULL       | 1               | Amit              
102         | Priya      | NULL       | 1               | Priya             
107         | Vikram     | NULL       | 1               | Vikram            
104         | Sneha      | 102        | 2               | Priya->Sneha      
106         | Neha       | 102        | 2               | Priya->Neha       
110         | Meera      | 102        | 2               | Priya->Meera      
109         | Karan      | 106        | 3               | Priya->Neha->Karan
103         | Rahul      | 101        | 2               | Amit->Rahul       
105         | Arjun      | 101        | 2               | Amit->Arjun       
108         | Pooja      | 105        | 3               | Amit->Arjun->Pooja
(10 rows)

Total execution time: 00:00:01.024